In [1]:
import xarray as xr
import numpy as np
from scipy.interpolate import griddata
import os
import geopandas as gpd
import rasterio
from rasterio import mask
import rioxarray
from rasterio.io import MemoryFile
from shapely.geometry import Polygon

def process_file(input_path, output_folder):
    # Format the datetime string
    ts = input_path.split("/")[-1].split(".")[2].split('-')[-1]
    date = input_path.split("/")[-1].split(".")[2].split('-')[-2][1:]
    formatted_datetime = f"{date[:4]}-{date[4:6]}-{date[6:]}T{ts[:2]}:{ts[2:4]}:{ts[4:]}Z"
    
    # Open the NetCDF file
    xds = xr.open_dataset(input_path)
    xds = xds.assign_coords(lon=(((xds.lon + 180) % 360) - 180)).sortby("lon")

    # Extract lat, lon, and wind_speed
    lat = xds['lat'].data
    lon = xds['lon'].data
    wind_speed = xds['wind_speed']

    print("here",lat.shape, lon.shape)

    # Ensure the dataset has the proper spatial dimensions and CRS
    wind_speed.rio.set_spatial_dims("lon", "lat", inplace=True)
    wind_speed.rio.write_crs("epsg:4326", inplace=True)

    # Define AOI as a shapely Polygon
    aoi_coords = [[-102.8148701375, 6.1943456775], [-13.3448605043, 6.1943456775], 
                  [-13.3448605043, 49.6429910636], [-102.8148701375, 49.6429910636], 
                  [-102.8148701375, 6.1943456775]]
    aoi_polygon = Polygon(aoi_coords)
    aoi_gdf = gpd.GeoDataFrame({'geometry': [aoi_polygon]}, crs="EPSG:4326")
    
    # Clip the data to the AOI
    wind_speed_clipped = wind_speed.rio.clip(aoi_gdf.geometry, aoi_gdf.crs, drop=True)
    print("After clipping", wind_speed.shape,wind_speed_clipped.shape)
    
    for i, timestamp in enumerate(wind_speed_clipped.time.values):
        time_slice = wind_speed_clipped.isel(time=i) 
        time_str = str(time_slice.time.values)[:19] +"Z"
        print("extracted_time", time_str)
        
        # Prepare output file path
        filename = f"cyg_wind_{time_str}.tif"
        output_path = os.path.join(output_folder, filename)
    
        with MemoryFile() as memfile:
            with memfile.open(driver='GTiff',
                              height=time_slice.shape[0],
                              width=time_slice.shape[1],
                              count=1,
                              dtype=time_slice.dtype,
                              crs="EPSG:4326",
                              transform=time_slice.rio.transform()) as dataset:
                dataset.write(time_slice.data, 1)
    
            # Mask the raster with the GeoJSON
            geo = gpd.read_file("cyg/land_mask.geojson")
            with memfile.open() as src:
                out_image, out_transform = mask.mask(src, geo.geometry, filled=True, invert=True)
                out_meta = src.meta.copy()
        out_image[out_image == 0] = -9999
        out_image = np.nan_to_num(out_image, nan=-9999)

    
        # Update the metadata to reflect the new dimensions
        out_meta.update({
            "driver": "COG",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "nodata": -9999
        })
        print(out_image.shape)
        
        with rasterio.open(output_path, "w", **out_meta) as dest:
            dest.write(out_image)
        

# Input and output folders
input_folder = "cyg/input"
output_folder = "cyg/output"

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

# Process all NetCDF files in the input folder
for file_name in os.listdir(input_folder):
    if file_name.endswith(".nc"):
        input_path = os.path.join(input_folder, file_name)
        process_file(input_path, output_folder)

print("Processing completed.")